In [ ]:
# General library imports.
import sys
from pathlib import Path
from itertools import islice
from collections import Counter
import random
from IPython.display import display
from datasets import load_dataset, Image

# Make project modules available when running from the notebook directory.
project_dir = str(Path.cwd().parent)
if project_dir not in sys.path:
    sys.path.append(project_dir)

# Reload local modules during interactive use.
%load_ext autoreload
%autoreload 2
%aimport -torch -sklearn -numpy -np -pandas -tensorflow

REPO_ID = "cvl-umass/RealBirdID"
REVISION = "abd12f9a663f47bc64b8345eeb54679d134762aa"  # Use a commit hash to reproduce a specific release.


In [ ]:
# Stream a small sample without downloading the full benchmark.
answerable = load_dataset(REPO_ID, "answerable", split="validation", revision=REVISION, streaming=True)
unanswerable = load_dataset(REPO_ID, "unanswerable", split="validation", revision=REVISION, streaming=True)


In [ ]:
# Disable image decoding while inspecting metadata.
rows = list(islice(answerable.cast_column("image", Image(decode=False)), 5))
assert rows
assert all(row["license"] for row in rows)
print("Columns:", sorted(rows[0]))
for row in rows:
    print(row["observation_id"], row["taxon_name"], row["license"], row["source_url"])
print("Sample licenses:", Counter(row["license"] for row in rows))


In [ ]:
# Inspect unanswerable labels and reasons without loading all images.
for row in islice(unanswerable.cast_column("image", Image(decode=False)), 3):
    print(row["observation_id"], row["taxon_name"], row["reasons"], row["license"])

# The configs are validation sets. There is no train split.
# Increase the sample size above to inspect more rows.


In [ ]:
# Select only rows whose photos have a Creative Commons license.
cc_licenses = {"cc0", "cc-by", "cc-by-sa", "cc-by-nd", "cc-by-nc", "cc-by-nc-sa", "cc-by-nc-nd"}
cc_only_unanswerable = unanswerable.filter(lambda row: row["license"] in cc_licenses)
print("First CC-only example:", next(iter(cc_only_unanswerable))["observation_id"])
# Non-CC rows have license == "all-rights-reserved" or "unknown".
# Their source_license_code remains None; inclusion grants no redistribution rights.
